**Geochemistry Biplot App for Bruker Results.csv Files**

Jupyter Notebook Version
N. Tripcevich 2026, CC BY-SA 4.0  
[More Information Online](https://github.com/arf-berkeley/bruker-xrf-ppm-plot)

For basic use click here, then proceed through this notebook cell-by-cell by pressing Shift-Return on your keyboard. Follow the instructions provided to upload your .csv file and view the data.

The Python in this live notebook can be edited and run again.

In [1]:
%%capture
%pip install plotly ipywidgets
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.colors import DEFAULT_PLOTLY_COLORS
from IPython.display import display
import ipywidgets as widgets
import io

**Select the Results.csv table from your Bruker analysis**

Browse to a copy of the __Results.csv__ file typically found in Bruker/Data/Results.csv

The code below imports Method Weight % results from the most recent analysis from the end of the Results.csv file.

In [ ]:
# Cell 2 - CSV Import
# Supports: VS Code (local) | Binder | Voila
from io import StringIO
import functools
import os
import sys

# ── Constants ─────────────────────────────────────────────────────────────
ALL_APPS    = 'All applications'
ALL_BATCHES = 'All'
LOD_VALUES  = {'< LOD': None, 'None': None, '': None}

# ── State container ───────────────────────────────────────────────────────
class StudyState:
    raw      = None
    filtered = None

state = StudyState()

# ── Master UI container (pre-displayed for Voila) ─────────────────────────
main_out = widgets.Output()
display(main_out)

# ── Environment detection ──────────────────────────────────────────────────
@functools.lru_cache(maxsize=None)
def is_local():
    hosted_signals = [
        'BINDER_LAUNCH_HOST',
        'JUPYTERHUB_USER',
        'JUPYTERHUB_SERVICE_PREFIX',
        'COLAB_BACKEND_VERSION',
        'VOILA_APP_PORT',
        'SERVER_SOFTWARE',
    ]
    if any(os.environ.get(v) for v in hosted_signals):
        return False
    if 'voila' in sys.modules:
        return False
    try:
        import tkinter as tk
        root = tk.Tk()
        root.withdraw()
        root.destroy()
        return True
    except Exception:
        return False

# ── Parser output widget ───────────────────────────────────────────────────
parse_out = widgets.Output()

# ── Parser ────────────────────────────────────────────────────────────────
def parse_results_csv(content_str, verbose=True):
    def _print(*args):
        if not verbose:
            return
        with parse_out:
            print(*args)

    content_str = content_str.replace('\r\n', '\n').replace('\r', '\n')
    lines       = [l for l in content_str.split('\n') if l.strip()]

    segment_starts = [
        i for i, line in enumerate(lines)
        if line.split(',')[0].strip().strip('"') == 'File #'
    ]

    if not segment_starts:
        raise ValueError('No "File #" header row found — is this a Bruker Results.csv?')

    frames = []
    for idx, start in enumerate(segment_starts, start=1):
        end           = segment_starts[idx] if idx < len(segment_starts) else len(lines)
        segment_lines = lines[start:end]
        if len(segment_lines) < 2:
            continue
        try:
            df_seg = pd.read_csv(
                StringIO('\n'.join(segment_lines)),
                dtype=str,
                skipinitialspace=True
            )
        except Exception as e:
            _print(f'  Segment {idx} skipped: {e}')
            continue

        df_seg.dropna(how='all', inplace=True)
        df_seg.dropna(axis=1, how='all', inplace=True)
        if df_seg.empty:
            continue

        df_seg['_batch'] = idx
        frames.append(df_seg)

    if not frames:
        raise ValueError('No data found after parsing all segments.')

    df = pd.concat(frames, ignore_index=True, join='outer')
    df.replace(LOD_VALUES, inplace=True)

    apps = (
        df['Application'].dropna().unique().tolist()
        if 'Application' in df.columns else []
    )
    _print(f'✓ Loaded {len(df)} rows | '
           f'{df["_batch"].nunique()} segments | '
           f'Applications: {apps}')
    return df

# ── Helper: normalize Application column ──────────────────────────────────
def _app_str(df):
    return df['Application'].astype(object).fillna('').astype(str).str.strip()

# ── Upload decoder (handles ipywidgets v7, v8, memoryview) ────────────────
def _decode_upload(upload_widget):
    val = upload_widget.value
    if isinstance(val, dict):
        if not val:
            raise ValueError('No file uploaded.')
        content_data = next(iter(val.values()))['content']
    elif isinstance(val, (list, tuple)):
        if not val:
            raise ValueError('No file uploaded.')
        content_data = val[0]['content']
    else:
        raise ValueError(f'Unrecognised FileUpload value type: {type(val)}')

    if isinstance(content_data, memoryview):
        raw = bytes(content_data)
    elif isinstance(content_data, (bytes, bytearray)):
        raw = bytes(content_data)
    else:
        raw = content_data.tobytes()

    return raw.decode('utf-8', errors='replace')

# ── Filter UI ─────────────────────────────────────────────────────────────
def build_filter_ui(host_out):
    if state.raw is None:
        with host_out:
            print('No data loaded yet.')
        return

    state.filtered = state.raw.copy()

    def get_unique(col):
        if col not in state.raw.columns:
            return []
        return sorted(
            state.raw[col]
            .dropna()
            .astype(str)
            .str.strip()
            .replace('', np.nan)
            .dropna()
            .unique()
            .tolist()
        )

    def batches_for(application):
        df = state.raw.copy()
        if application != ALL_APPS:
            df = df[_app_str(df) == application.strip()]
        return [ALL_BATCHES] + [
            str(b) for b in sorted(df['_batch'].dropna().unique())
        ]

    all_applications = [ALL_APPS] + get_unique('Application')
    default_app      = all_applications[1] if len(all_applications) > 1 else ALL_APPS

    app_dd = widgets.Dropdown(
        options=all_applications,
        value=default_app,
        description='Application:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='360px')
    )
    batch_dd = widgets.Dropdown(
        options=batches_for(default_app),
        value=ALL_BATCHES,
        description='Batch:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='180px')
    )
    apply_btn = widgets.Button(
        description='Apply Filter',
        button_style='primary',
        icon='filter',
        layout=widgets.Layout(width='150px')
    )
    summary_out = widgets.Output()
    filter_out  = widgets.Output()

    def refresh_summary(application):
        with summary_out:
            summary_out.clear_output(wait=True)
            df    = state.raw.copy()
            if application != ALL_APPS:
                df = df[_app_str(df) == application.strip()]
            batches = sorted(df['_batch'].unique().tolist())
            dates   = pd.to_datetime(df['DateTime'], errors='coerce').dropna()
            d_min   = dates.min().strftime('%m-%d-%Y') if not dates.empty else '?'
            d_max   = dates.max().strftime('%m-%d-%Y') if not dates.empty else '?'
            print(f'  "{application}" -> {len(df)} rows')
            print(f'  Batch(es) : {batches}')
            print(f'  Date range: {d_min} - {d_max}')

    def on_app_change(change):
        new_opts         = batches_for(change['new'])
        batch_dd.options = new_opts
        batch_dd.value   = ALL_BATCHES
        refresh_summary(change['new'])

    def on_apply(btn):
        filter_out.clear_output(wait=True)
        with filter_out:
            df        = state.raw.copy()
            sel_app   = app_dd.value
            sel_batch = batch_dd.value

            if sel_app != ALL_APPS:
                df = df[_app_str(df) == sel_app.strip()]

            if sel_batch != ALL_BATCHES:
                try:
                    batch_val = int(sel_batch)
                except ValueError:
                    print(f'Invalid batch value: {sel_batch}')
                    return
                df = df[df['_batch'] == batch_val]

            df             = df.reset_index(drop=True)
            state.filtered = df

            if df.empty:
                print(f'No rows for application="{sel_app}" batch="{sel_batch}"')
                print(f'Applications in file : {get_unique("Application")}')
                print(f'Batches in file      : '
                      f'{sorted(state.raw["_batch"].unique().tolist())}')
                return

            dates = pd.to_datetime(df['DateTime'], errors='coerce').dropna()
            d_min = dates.min().strftime('%m-%d-%Y %H:%M') if not dates.empty else '?'
            d_max = dates.max().strftime('%m-%d-%Y %H:%M') if not dates.empty else '?'

            try:
                file_nums = df['File #'].dropna().astype(int)
                f_min     = int(file_nums.min())
                f_max     = int(file_nums.max())
            except Exception:
                f_min = df['File #'].dropna().min() if 'File #' in df.columns else '?'
                f_max = df['File #'].dropna().max() if 'File #' in df.columns else '?'

            print(f'✓ {len(df)} rows kept')
            print(f'  Application : {sel_app}')
            print(f'  Batch       : {sel_batch}')
            print(f'  File # range: {f_min} - {f_max}')
            print(f'  Date range  : {d_min} - {d_max}')
            print('\nNow run Cell 3 -> Cell 4 -> Cell 5')

    app_dd.observe(on_app_change, names='value')
    apply_btn.on_click(on_apply)

    with host_out:
        host_out.clear_output(wait=True)
        display(parse_out)
        display(widgets.HTML('<b>Filter by Application and Batch</b>'))
        display(widgets.HTML(
            '<span style="color:grey;font-size:12px">'
            'Choose an Application - Batch will update automatically. '
            'Then click Apply Filter.</span>'
        ))
        display(widgets.VBox([
            widgets.HBox([app_dd, batch_dd, apply_btn]),
            summary_out,
            filter_out
        ]))

    refresh_summary(app_dd.value)

# ── File loading - VS Code / local ────────────────────────────────────────
def load_local():
    import tkinter as tk
    from tkinter import filedialog

    if state.raw is not None:
        with main_out:
            print(f'Already loaded: {state.raw.shape[0]} rows')
        build_filter_ui(main_out)
        return

    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    study_path = filedialog.askopenfilename(
        title='Select Results.csv',
        filetypes=[('CSV files', '*.csv'), ('All files', '*.*')]
    )
    root.destroy()

    with main_out:
        if study_path:
            try:
                with open(study_path, 'r', encoding='utf-8', errors='replace') as f:
                    state.raw = parse_results_csv(f.read())
                print(f'  {study_path}\n')
                build_filter_ui(main_out)
            except Exception as e:
                print(f'Failed to load file: {e}')
        else:
            print('No file selected - re-run this cell to try again.')

# ── File loading - Binder / Voila / hosted ────────────────────────────────
def load_hosted():
    if state.raw is not None:
        with main_out:
            print(f'Already loaded: {state.raw.shape[0]} rows')
        build_filter_ui(main_out)
        return

    upload_widget = widgets.FileUpload(accept='.csv', multiple=False)
    load_btn      = widgets.Button(
        description='Load Data',
        button_style='success',
        icon='check',
        disabled=True
    )
    status_lbl  = widgets.Label(
        'Upload a Bruker XRF Results.csv file'
    )
    session_lbl = widgets.HTML(
        '<span style="color:grey;font-size:12px">'
        'Sessions are temporary - you will need to re-upload each session.'
        '</span>'
    )
    filter_area = widgets.Output()

    def _on_upload_change(change):
        if upload_widget.value:
            load_btn.disabled = False
            try:
                val = upload_widget.value
                if isinstance(val, dict):
                    size = next(iter(val.values())).get('size', 0)
                else:
                    size = len(val[0].get('content', b''))
                size_mb = size / (1024 * 1024)
                if size_mb > 50:
                    status_lbl.value = (
                        f'Large file ({size_mb:.1f} MB) - '
                        f'may be slow. Click Load Data.'
                    )
                else:
                    status_lbl.value = (
                        f'File ready ({size_mb:.1f} MB) - click Load Data'
                    )
            except Exception:
                status_lbl.value = 'File ready - click Load Data'
        else:
            load_btn.disabled = True

    def _on_load(btn):
        try:
            content              = _decode_upload(upload_widget)
            state.raw            = parse_results_csv(content)
            status_lbl.value     = f'✓ Loaded {state.raw.shape[0]} rows'
            load_btn.disabled    = True
            load_btn.description = 'Loaded'
            with filter_area:
                filter_area.clear_output(wait=True)
                build_filter_ui(filter_area)
        except Exception as e:
            status_lbl.value = f'Error: {e}'

    upload_widget.observe(_on_upload_change, names='value')
    load_btn.on_click(_on_load)

    with main_out:
        display(widgets.VBox([
            widgets.Label('Upload Results.csv:'),
            upload_widget,
            load_btn,
            status_lbl,
            session_lbl,
            filter_area
        ]))

# ── Entry point ───────────────────────────────────────────────────────────
if is_local():
    load_local()
else:
    load_hosted()

Output()

***[Click Here to Continue]***

__Clean up Bruker data__

Cleaning data includes removing the following: elemental error columns, Alloy, Match Qual columns, Multiplier, Cal Check, Operator, Field 1&2. This script also replaces Below Detection Limits LOD with 0.

In [ ]:
# Cell 3 - Data Cleaning
# Supports: VS Code (local) | Binder | Voila
import re

# ── Constants (add these to Cell 2 alongside LOD_VALUES) ──────────────────
NON_ELEMENT_COLS = [
    'Alloy 1', 'Match Qual 1', 'Alloy 2', 'Match Qual 2',
    'Alloy 3', 'Match Qual 3', 'Multiplier', 'Cal Check',
    'Operator', 'Field1', 'Field2', 'ID', '_batch'
]

# Bruker element columns follow standard chemical symbol pattern
ELEMENT_PATTERN = re.compile(r'^[A-Z][a-z]?$')

# ── Output widget (visible in Voila) ──────────────────────────────────────
cell3_out = widgets.Output()
display(cell3_out)

with cell3_out:
    # ── Guard ─────────────────────────────────────────────────────────────
    if state.filtered is None or len(state.filtered) == 0:
        print('Please apply a filter in Cell 2 before continuing.')
    else:
        _source = state.filtered.copy()
        print(f'Using filtered data : {len(_source)} rows')
        print(f'Application(s)      : '
              f'{_source["Application"].dropna().unique().tolist()}')
        print(f'Batch(es)           : '
              f'{sorted(_source["_batch"].unique().tolist())}')

        # ── Drop non-element columns ───────────────────────────────────────
        # Drop error columns (contain 'Err') and known non-element columns
        drop_cols = [
            c for c in _source.columns
            if 'Err' in c or c in NON_ELEMENT_COLS
        ]
        keep_cols = [c for c in _source.columns if c not in drop_cols]
        study     = _source[keep_cols].copy()

        # ── Identify column types ──────────────────────────────────────────
        # Known Bruker string columns
        string_cols = [
            c for c in ['Name', 'Application', 'Method']
            if c in study.columns
        ]
        # Known non-element numeric columns
        meta_numeric = ['File #', 'ElapsedTime']

        # Element columns: match chemical symbol pattern (Fe, Cu, Pb, etc.)
        element_cols = [
            c for c in study.columns
            if ELEMENT_PATTERN.match(c)
            and c not in string_cols
            and c not in meta_numeric
            and c != 'DateTime'
        ]
        # All numeric columns including meta
        numeric_cols = [
            c for c in study.columns
            if c not in string_cols + ['DateTime']
        ]

        # ── Type casting ───────────────────────────────────────────────────
        study[string_cols]  = study[string_cols].astype('string')
        study[numeric_cols] = study[numeric_cols].apply(
            pd.to_numeric, errors='coerce'
        )
        study['DateTime'] = pd.to_datetime(study['DateTime'], errors='coerce')

        # ── Scale element concentrations to ppm ────────────────────────────
        # Bruker exports fractional concentrations (0-1 range)
        # Multiply by 10000 to convert to ppm
        study[element_cols] = (study[element_cols] * 10000).round(1)

        # ── Handle below-detection values ──────────────────────────────────
        # Cell 2 already replaced '< LOD' with None/NaN
        # Fill remaining NaN in element columns with 0
        # (below detection limit treated as zero concentration)
        study[element_cols] = study[element_cols].fillna(0)

        # ── Drop empty rows and columns ────────────────────────────────────
        # Drop all-NaN columns first, then all-NaN rows
        # (column drop first prevents valid rows being lost)
        study.dropna(axis=1, how='all', inplace=True)
        study.dropna(how='all', inplace=True)
        study = study.reset_index(drop=True)

        # ── Store on state ─────────────────────────────────────────────────
        state.study = study

        # ── Summary ────────────────────────────────────────────────────────
        print(f'\nRows after cleaning : {len(study)}')
        print(f'Element columns     : {element_cols}')
        print(f'All columns         : {study.columns.tolist()}')
        print('\nNow run Cell 4 -> Cell 5')

Using filtered data : 68 rows
Application(s)      : ['Obsidian 3mm']
Batch(es)           : [14]
Rows after cleaning : 68
Columns             : ['File #', 'DateTime', 'Application', 'Method', 'ElapsedTime', 'Mn', 'Zr', 'Rb', 'Sr', 'Y', 'Nb', 'Ba', 'Th']


**Display Data Table**

Run the next cell to view the data table before viewing a biplot.

In [4]:
# Cell 4 

if 'uploader' in globals() and upload_widget.value:
    # Handle both old (tuple/dict) and new (list) ipywidgets versions
    if isinstance(upload_widget.value, (list, tuple)):
        file_info = upload_widget.value[0]
    else:
        file_info = list(upload_widget.value.values())[0]
        
    content = file_info['content']
else:
    print("Please upload a file in Cell 3 first.")

def parse_bruker_vertical(content):
    samples = []
    current_sample = {}
    
    for line in content.splitlines():
        line = line.strip()
        if not line or ':' not in line:
            continue
            
        parts = line.split(':', 1)
        k, v = parts[0].strip(), parts[1].strip()
        
        # Skip Bruker's repeated header rows (e.g. "Ti: Ti")
        if k == v or v in ["DateTime", "File #", "Multiplier"]:
            continue
            
        # Start new sample block
        if k == "File #":
            if current_sample:
                samples.append(current_sample)
            current_sample = {}
        
        # Clean numeric strings
        if v == "< LOD" or v == "1.#SNB" or v == "" or v == "None":
            v = 0.0
        elif v == "Passed":
            v = 1.0
        else:
            try:
                # Keep numeric values as floats
                v = float(v)
            except ValueError:
                pass # Keep metadata like 'Application' as string
        
        current_sample[k] = v
        
    if current_sample:
        samples.append(current_sample)
        
    df_raw = pd.DataFrame(samples)
    
    # --- CRITICAL FIX FOR CELL 6 ---
    # 1. Fill all missing values with 0.0 (prevents plotting errors)
    df_raw = df_raw.fillna(0.0)
    
    # 2. Ensure element columns are actually numeric
    # Metadata columns to exclude from element list
    meta = ['File #', 'DateTime', 'Operator', 'Name', 'ID', 'Field1', 'Field2', 'Application', 'Method', 'ElapsedTime', 'Multiplier', 'Cal Check']
    
    for col in df_raw.columns:
        if col not in meta and 'Err' not in col:
            df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce').fillna(0.0)
            
    return df_raw

# Execute uploader logic
if upload_widget.value:
    file_info = list(upload_widget.value.values())[0]
    content = file_info['content']
    if isinstance(content, bytes):
        content = content.decode('utf-8', errors='ignore')
        
    df = parse_bruker_vertical(content)
    
    # Re-identify available elements for Cell 5 dropdowns
    meta = ['File #', 'DateTime', 'Operator', 'Name', 'ID', 'Field1', 'Field2', 'Application', 'Method', 'ElapsedTime', 'Multiplier', 'Cal Check']
    el_options = sorted([c for c in df.columns if c not in meta and 'Err' not in c])
    
    print(f"Ready: {len(df)} samples parsed. Use dropdowns below.")


Please upload a file in Cell 3 first.


NameError: name 'upload_widget' is not defined

In [ ]:
# Cell 5 - Biplot
if study is None:
    print('Please complete data cleaning before continuing')
else:
    NON_ELEMENT_COLS = ['File #', 'DateTime', 'Name', 'Application',
                        'Method', 'ElapsedTime', 'Elapsed', '_batch', '_method']
    elements_present = [
        c for c in study.columns
        if c not in NON_ELEMENT_COLS
        and pd.api.types.is_numeric_dtype(study[c])
    ]

    if len(elements_present) < 2:
        print(f'Not enough numeric element columns to plot. Found: {elements_present}')
    else:
        x_dropdown = widgets.Dropdown(
            options=elements_present,
            value='Sr' if 'Sr' in elements_present else elements_present[0],
            description='X Axis:',
            style={'description_width': 'initial'}
        )
        y_dropdown = widgets.Dropdown(
            options=elements_present,
            value='Rb' if 'Rb' in elements_present else elements_present[1],
            description='Y Axis:',
            style={'description_width': 'initial'}
        )

        plot_output = widgets.Output()

        def update_plot(change):
            with plot_output:
                plot_output.clear_output(wait=True)
                x = x_dropdown.value
                y = y_dropdown.value

                # Make a plain-Python-string copy of study for plotly
                # This avoids the pandas NA ambiguous boolean error
                plot_df = study.copy()

                # Convert all string/object/StringDtype columns to plain str
                for col in plot_df.columns:
                    if hasattr(plot_df[col], 'dtype') and (
                        plot_df[col].dtype == 'string' or
                        plot_df[col].dtype == object or
                        str(plot_df[col].dtype) == 'StringDtype'
                    ):
                        plot_df[col] = plot_df[col].astype(object).where(
                            plot_df[col].notna(), other='(no name)'
                        ).astype(str)

                # Fill any remaining NA in Name
                if 'Name' in plot_df.columns:
                    plot_df['Name'] = plot_df['Name'].replace(
                        {'nan': '(no name)', 'None': '(no name)', '<NA>': '(no name)'}
                    ).fillna('(no name)')

                color_col  = 'Name' if 'Name' in plot_df.columns else None
                name_order = (
                    sorted(plot_df['Name'].unique().tolist())
                    if color_col else []
                )

                # Build hover list from columns that exist and aren't x or y
                hover_data = [
                    c for c in ['File #', 'DateTime']
                    if c in plot_df.columns and c != x and c != y
                ]

                try:
                    fig = px.scatter(
                        plot_df,
                        x=x,
                        y=y,
                        color=color_col,
                        category_orders={'Name': name_order},
                        hover_data=hover_data,
                        title=f'{y} vs {x} Biplot',
                        labels={
                            x: f'{x} (PPM)',
                            y: f'{y} (PPM)',
                            'Name': 'Sample Name'
                        }
                    )
                    fig.update_traces(marker=dict(size=8, opacity=0.85))
                    fig.update_layout(
                        height=600,
                        hovermode='closest',
                        legend=dict(
                            title=dict(text='Sample Name', font=dict(size=13)),
                            itemsizing='constant',
                            bordercolor='lightgrey',
                            borderwidth=1,
                            bgcolor='rgba(255,255,255,0.85)',
                            x=1.02,
                            xanchor='left',
                            y=1,
                            yanchor='top'
                        ),
                        margin=dict(r=180)
                    )
                    fig.show()
                except Exception as e:
                    print(f'Plot error: {e}')
                    print(f'  x={x}, y={y}')
                    print(f'  Name dtype: {plot_df["Name"].dtype if "Name" in plot_df.columns else "missing"}')
                    print(f'  Name sample: {plot_df["Name"].unique()[:5] if "Name" in plot_df.columns else "n/a"}')

        x_dropdown.observe(update_plot, names='value')
        y_dropdown.observe(update_plot, names='value')

        display(widgets.VBox([
            widgets.HBox([x_dropdown, y_dropdown]),
            plot_output
        ]))

        update_plot(None)

In [ ]:
# Cell 6 - Ternary Plot



import traceback
import sys

# We create a special output area to catch the error
debug_out = widgets.Output()
display(debug_out)

with debug_out:
    try:
        # --- ORIGINAL PLOTTING LOGIC STARTS HERE ---
        # (This section tries to run your actual plot)
        
        # 1. Check if df exists
        if 'df' not in locals() or df.empty:
            print("Error: Dataframe 'df' is empty or not defined. Did Cell 4 run successfully?")
        
        # 2. Check if widgets exist
        elif 'x_axis' not in locals() or 'y_axis' not in locals():
            print("Error: Dropdown widgets not found. Did Cell 5 run successfully?")
            
        else:
            # This is the line that usually crashes:
            # We wrap it in a helper to ensure it's numeric just in case
            x_col = x_axis.value
            y_col = y_axis.value
            
            print(f"Attempting to plot: {x_col} vs {y_col}")
            
            # --- START PLOT ---
            # Note: If your script uses plotly (px.scatter) or matplotlib (plt.scatter), 
            # make sure the code below matches your library.
            
            import matplotlib.pyplot as plt
            
            plt.figure(figsize=(10, 6))
            plt.scatter(
                pd.to_numeric(df[x_col], errors='coerce'), 
                pd.to_numeric(df[y_col], errors='coerce'),
                alpha=0.5
            )
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.title(f"{x_col} vs {y_col}")
            plt.show()
            # --- END PLOT ---

    except Exception as e:
        # This catches the error and prints it to the screen in the dashboard
        print("!!! CELL 6 CRASHED !!!")
        print(f"Error Type: {type(e).__name__}")
        print(f"Error Message: {str(e)}")
        print("\n--- Full Traceback below ---")
        traceback.print_exc(file=sys.stdout)
        
        
        
    if study is None:
    print('Please complete data cleaning before continuing')
else:
    NON_ELEMENT_COLS = ['File #', 'DateTime', 'Name', 'Application',
                        'Method', 'ElapsedTime', 'Elapsed', '_batch', '_method']
    elements_present = [
        c for c in study.columns
        if c not in NON_ELEMENT_COLS
        and pd.api.types.is_numeric_dtype(study[c])
    ]

    if len(elements_present) < 3:
        print(f'Not enough numeric element columns. Found: {elements_present}')
    else:
        # Default to Rb, Sr, Zr if available
        default_a = 'Rb' if 'Rb' in elements_present else elements_present[0]
        default_b = 'Sr' if 'Sr' in elements_present else elements_present[1]
        default_c = 'Zr' if 'Zr' in elements_present else elements_present[2]

        a_dropdown = widgets.Dropdown(
            options=elements_present,
            value=default_a,
            description='A (top):',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='200px')
        )
        b_dropdown = widgets.Dropdown(
            options=elements_present,
            value=default_b,
            description='B (bottom left):',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='220px')
        )
        c_dropdown = widgets.Dropdown(
            options=elements_present,
            value=default_c,
            description='C (bottom right):',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='220px')
        )

        plot_output = widgets.Output()

        def update_plot(change):
            with plot_output:
                plot_output.clear_output(wait=True)
                a = a_dropdown.value
                b = b_dropdown.value
                c = c_dropdown.value

                if len({a, b, c}) < 3:
                    print('Please select three different elements.')
                    return

                # Clean copy with plain Python strings for plotly
                plot_df = study.copy()
                for col in plot_df.columns:
                    if str(plot_df[col].dtype) in ('string', 'StringDtype') or \
                       plot_df[col].dtype == object:
                        plot_df[col] = (
                            plot_df[col]
                            .astype(object)
                            .fillna('(no name)')
                            .astype(str)
                        )

                if 'Name' in plot_df.columns:
                    plot_df['Name'] = plot_df['Name'].replace(
                        {'nan': '(no name)', 'None': '(no name)', '<NA>': '(no name)'}
                    )

                # Drop rows where any of the three elements are NaN
                plot_df = plot_df.dropna(subset=[a, b, c])

                if plot_df.empty:
                    print(f'No rows with valid data for {a}, {b}, {c}.')
                    return

                color_col  = 'Name' if 'Name' in plot_df.columns else None
                name_order = (
                    sorted(plot_df['Name'].unique().tolist())
                    if color_col else []
                )

                try:
                    fig = px.scatter_ternary(
                        plot_df,
                        a=a,
                        b=b,
                        c=c,
                        color=color_col,
                        category_orders={'Name': name_order},
                        hover_data=[
                            col for col in ['File #', 'DateTime']
                            if col in plot_df.columns
                        ],
                        title=f'Ternary Plot: {a} / {b} / {c}',
                        labels={
                            'Name': 'Sample Name',
                            a: f'{a} (PPM)',
                            b: f'{b} (PPM)',
                            c: f'{c} (PPM)'
                        }
                    )
                    fig.update_traces(marker=dict(size=8, opacity=0.85))
                    fig.update_layout(
                        height=650,
                        legend=dict(
                            title=dict(text='Sample Name', font=dict(size=13)),
                            itemsizing='constant',
                            bordercolor='lightgrey',
                            borderwidth=1,
                            bgcolor='rgba(255,255,255,0.85)',
                            x=1.02,
                            xanchor='left',
                            y=1,
                            yanchor='top'
                        ),
                        margin=dict(r=180)
                    )
                    fig.show()
                except Exception as e:
                    print(f'Plot error: {e}')
                    print(f'  a={a}, b={b}, c={c}')
                    print(f'  Name dtype: {plot_df["Name"].dtype if "Name" in plot_df.columns else "missing"}')

        a_dropdown.observe(update_plot, names='value')
        b_dropdown.observe(update_plot, names='value')
        c_dropdown.observe(update_plot, names='value')

        display(widgets.VBox([
            widgets.HBox([a_dropdown, b_dropdown, c_dropdown]),
            plot_output
        ]))

        update_plot(None)

In [ ]:
# Cell 7 - Export dataset as CSV
from IPython.display import display, HTML
if study is None:
    print('Please complete data cleaning before continuing')
else:
    export_output = widgets.Output()

    export_btn = widgets.Button(
        description='Export CSV',
        button_style='success',
        icon='download'
    )

    def on_export(btn):
        with export_output:
            export_output.clear_output(wait=True)
            if is_local():
                import tkinter as tk
                from tkinter import filedialog

                root = tk.Tk()
                root.withdraw()
                root.attributes('-topmost', True)
                save_path = filedialog.asksaveasfilename(
                    title='Save CSV',
                    defaultextension='.csv',
                    filetypes=[('CSV files', '*.csv'), ('All files', '*.*')],
                    initialfile=f'Bruker_Results_export_{study["DateTime"].max().strftime("%Y%m%d")}.csv'
                )
                root.destroy()

                if save_path:
                    study.to_csv(save_path, index=False)
                    print(f'✓ Exported {len(study)} rows to {save_path}')
                else:
                    print('Export cancelled')
            else:
                import base64
                from IPython.display import HTML
                csv_str = study.to_csv(index=False)
                b64 = base64.b64encode(csv_str.encode()).decode()
                filename = f'Bruker_Results_export_{study["DateTime"].max().strftime("%Y%m%d")}.csv'
                html = f'<a download="{filename}" href="data:text/csv;base64,{b64}">Click here to download {filename}</a>'
                display(HTML(html))

    export_btn.on_click(on_export)

    display(widgets.HTML('<b>Would you like to export the cleaned up CSV of these values?</b>'))
    display(widgets.VBox([export_btn, export_output]))